# YouTube Slide Extractor
Download a YouTube video, extract a frame every N seconds, and get a PDF of slides.

In [ ]:
!pip install -U yt-dlp opencv-python-headless img2pdf -q

In [ ]:
import subprocess, cv2, tempfile
from google.colab import files
from img2pdf import convert

YOUTUBE_URL = input("Enter YouTube URL: ")
INTERVAL_SECONDS = int(input("Extract slides every (seconds): "))
QUALITY = 95  # JPEG quality (1-100)

print("\n📥 Downloading video...")
temp_dir = tempfile.mkdtemp()
subprocess.run(
    ["yt-dlp", "-f", "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]/b",
     "--merge-output-format", "mp4", "-o", f"{temp_dir}/video.mp4", YOUTUBE_URL],
    check=True,
)

cap = cv2.VideoCapture(f"{temp_dir}/video.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
duration = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps)
print(f"📹 Duration: {duration//60}:{duration%60:02d}")

print(f"🎬 Extracting frames every {INTERVAL_SECONDS} seconds...")
images = []
for t in range(0, duration, INTERVAL_SECONDS):
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ret, frame = cap.read()
    if ret:
        path = f"{temp_dir}/slide_{t//60:02d}_{t%60:02d}.jpg"
        cv2.imwrite(path, frame, [cv2.IMWRITE_JPEG_QUALITY, QUALITY])
        images.append(path)
        print(f"  ✅ Extracted at {t//60:02d}:{t%60:02d}")
cap.release()

if images:
    print(f"📄 Creating PDF with {len(images)} slides...")
    pdf_path = f"{temp_dir}/slides.pdf"
    with open(pdf_path, "wb") as f:
        f.write(convert(images))
    files.download(pdf_path)
    print("✅ Done! PDF downloaded.")
else:
    print("❌ No slides extracted")